# API Chat Agent - Hosted Agent 배포

외부 API(JSONPlaceholder) 호출을 통해 Hosted Agent에서 외부 네트워크 통신이 가능한지 테스트합니다.  
LLM 호출 없이 `https://jsonplaceholder.typicode.com` API를 호출하고 결과를 반환합니다.

**키워드별 API 호출:**
- `post` → `/posts/1`
- `user` → `/users/1`
- `todo` → `/todos/1`
- `comment` → `/comments/1`
- 기본 → `/posts/1`

---

## 0. 환경 설정

In [ ]:
%pip install --quiet python-dotenv "azure-ai-projects>=2.0.0" azure-identity

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
ACCOUNT_NAME = os.getenv("ACCOUNT_NAME")
ACR_NAME = os.getenv("ACR_NAME")
AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")

print(f"Subscription: {SUBSCRIPTION_ID[:8]}...")
print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Account Name: {ACCOUNT_NAME}")
print(f"Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT[:20]}...")
print(f"ACR: {ACR_NAME}")

## 1. Docker 이미지 빌드 & 푸시

> ⚠️ Apple Silicon 맥북에서는 `--platform=linux/amd64` 필수

In [ ]:
AGENT_NAME = "api-chat"
IMAGE_NAME = "api-chat"
IMAGE_TAG = "v1"
FULL_IMAGE = f"{ACR_NAME}.azurecr.io/{IMAGE_NAME}:{IMAGE_TAG}"

print("터미널에서 아래 명령을 실행하세요:\n")
print(f"az acr login --name {ACR_NAME}")
print(f"docker build --platform=linux/amd64 -t {FULL_IMAGE} .")
print(f"docker push {FULL_IMAGE}")

## 2. Hosted Agent 생성

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    HostedAgentDefinition,
    ProtocolVersionRecord,
    AgentProtocol,
)
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    endpoint=AZURE_AI_PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
    allow_preview=True,
)

agent = project.agents.create_version(
    agent_name=AGENT_NAME,
    description="External API call test agent using JSONPlaceholder",
    definition=HostedAgentDefinition(
        container_protocol_versions=[
            ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version="v1")
        ],
        cpu="1",
        memory="2Gi",
        image=FULL_IMAGE,
    ),
)

print(f"✅ Agent created: {agent.name}, version: {agent.version}")

## 3. Agent 테스트

In [ ]:
agent_info = project.agents.get(agent_name=AGENT_NAME)
print(f"Agent: {agent_info.name}")
print(f"Versions: {agent_info.versions}")

In [ ]:
openai_client = project.get_openai_client()

# 'todo' 키워드로 /todos/1 호출 테스트
stream = openai_client.responses.create(
    stream=True,
    input=[{"role": "user", "content": "todo 조회해줘"}],
    extra_body={"agent_reference": {"name": agent_info.name, "type": "agent_reference"}},
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

## 4. 정리

In [ ]:
project.agents.delete_version(agent_name=AGENT_NAME, agent_version=agent.version)
print(f"Agent version {agent.version} deleted.")